In [1]:
# core/engine/lib/llm/agents/_04_personal/test.ipynb

# Setup project path and log in as test user.
from neurocad.utils.testing import req
req.login_test()

=== ОТЛАДКА TESTING.PY ===
Текущая рабочая директория: /root/projects/aleksmir-dev/neurocad/neurocad/core/engine/lib/llm/agents/_04_personal
Путь к test/.env, который мы ищем: /root/projects/aleksmir-dev/neurocad/test/.env
APP_PORT из os.getenv: 8000
[TEST] Logging in as test...
[TEST] Login successful


True

In [43]:
# Stage 1: Brief dialog (simulating user answers).

history = []
user_answers = [
    "Хочу сделать личную страницу",
    "Меня зовут Алексей Смирнов, я разработчик",
    "Мои проекты: Нейрокад и Проверка Email",
    "Люблю синие и белые цвета"
]
user_answers = [
    "Хочу сделать страницу про каталог компьютеров"
]

for msg in user_answers:
    print(f"👤 Пользователь: {msg}")
    payload = {
        "prompt": msg,
        "stage": "brief",
        "history": history
    }
    res = req.post("/core/engine/lib/llm/agents/_04_personal/generate", payload)
    
    if res.get("success"):
        data = res["data"]
        if data.get("question"):
            print(f"🤖 Агент: {data['question']}")
            history.append({"role": "user", "content": msg})
            history.append({"role": "assistant", "content": data["question"]})
        elif data.get("ready"):
            print("🤖 Агент: Отлично, все данные собраны! Начинаю генерацию...")
            break
    else:
        print("❌ Ошибка:", res.get("error"))

👤 Пользователь: Хочу сделать страницу про каталог компьютеров
[TEST] POST http://127.0.0.1:8012/core/engine/lib/llm/agents/_04_personal/generate


[TEST] Status: 200
{
  "success": true,
  "data": {
    "question": "Отлично! Как вас зовут и чем вы занимаетесь?",
    "ready": false
  }
}
🤖 Агент: Отлично! Как вас зовут и чем вы занимаетесь?


In [44]:
# Stage 2: Generate base layout.

import json
from IPython.display import clear_output

brief_text = "Имя: Алексей Смирнов. Профессия: Разработчик. Проекты: Нейрокад, Проверка Email. Цвета: синие и белые."
brief_text = user_answers[0]
payload = {"prompt": brief_text, "stage": "base"}
res = req.post("/core/engine/lib/llm/agents/_04_personal/generate", payload)

if not res.get("success"):
    print("❌ Ошибка:", res.get("error"))
else:
    data = res["data"]
    current_html = data["html"]
    css_by_section = dict(data["css_by_section"])

    with open("state_html.txt", "w", encoding="utf-8") as f:
        f.write(current_html)
    with open("state_css.json", "w", encoding="utf-8") as f:
        json.dump(css_by_section, f, ensure_ascii=False, indent=2)

    full_css = "\n".join(css_by_section.values())
    with open("preview.html", "w", encoding="utf-8") as f:
        f.write(
            "<!DOCTYPE html><html><head><meta charset='utf-8'>"
            f"<style>{full_css}</style></head><body>{current_html}</body></html>"
        )

    clear_output(wait=True)
    print("✅ Макет сгенерирован и сохранён в preview.html")
    print("   CSS по секциям: " + ", ".join(f"{k}={len(v)}" for k, v in css_by_section.items()))

✅ Макет сгенерирован и сохранён в preview.html
   CSS по секциям: hero=848, projects=1291, footer=599


In [ ]:
# Stage 3: Add effects to ALL sections (hero, projects, footer) in one loop.

import json, os, re
from IPython.display import clear_output

COLOR_HINT = (
    "Цвета: спокойный синий (#2563eb, #3b82f6, #60a5fa) и белый. "
    "Можно светло-желый, светло-зеленый, светло-синий"
    "НЕ использовать ядовитый неоновый голубой (#22d3ee, #06b6d4, #67e8f9) "
    "НЕ использовать фиолетовый (#a855f7, #7c3aed). "
)
COLOR_HINT = (
    "Разноцветные светл-зеленые, кроме сиреневого"
)

SECTIONS = [
    ("hero",     "section"),
    ("projects", "section"),
    ("footer",   "footer"),
]

with open("state_html.txt", encoding="utf-8") as f:
    current_html = f.read()
with open("state_css.json", encoding="utf-8") as f:
    css_by_section = json.load(f)

for section, tag in SECTIONS:
    print(f"\n🎨 Генерирую эффекты для «{section}»...")

    pattern = rf"(<{tag} class=['\"]{section}['\"].*?</{tag}>)"
    match = re.search(pattern, current_html, re.DOTALL)
    if not match:
        print(f"❌ Не нашли секцию {section} — пропускаю")
        continue

    section_html = match.group(1)

    payload = {
        "prompt": f"Добавь эффекты к {section}. {COLOR_HINT}",
        "stage": "effects",
        "target_section": section,
        "current_html": section_html,
        "css_by_section": css_by_section,
    }
    res = req.post("/core/engine/lib/llm/agents/_04_personal/generate", payload)

    if not res.get("success"):
        print(f"❌ Ошибка для {section}: {res.get('error')}")
        continue

    new_section_html = res["data"]["html"]
    new_section_css = res["data"]["css"]

    current_html = current_html.replace(section_html, new_section_html)
    css_by_section[section] = (
        css_by_section[section]
        + f"\n\n/* effects: {section} */\n"
        + new_section_css
    )

    # Сохраняем после каждой секции — если следующая упадёт, предыдущая уже на диске
    with open("state_html.txt", "w", encoding="utf-8") as f:
        f.write(current_html)
    with open("state_css.json", "w", encoding="utf-8") as f:
        json.dump(css_by_section, f, ensure_ascii=False, indent=2)

    print(f"✅ Эффекты для {section} добавлены.")

# Финальный рендер
full_css = "\n".join(css_by_section.values())
with open("preview.html", "w", encoding="utf-8") as f:
    f.write(
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        f"<style>{full_css}</style></head><body>{current_html}</body></html>"
    )

clear_output(wait=True)
print("✅ Все эффекты добавлены. preview.html обновлён.")
print("   CSS по секциям: " + ", ".join(f"{k}={len(v)}" for k, v in css_by_section.items()))

✅ Все эффекты добавлены. preview.html обновлён.
   CSS по секциям: hero=3266, projects=3914, footer=1331


In [5]:
# Start a simple HTTP server to preview the generated page.

import http.server
import socketserver
import threading

PORT = 8898

Handler = http.server.SimpleHTTPRequestHandler
httpd = socketserver.TCPServer(("", PORT), Handler)

thread = threading.Thread(target=httpd.serve_forever, daemon=True)
thread.start()

print(f"Page available at: http://127.0.0.1:{PORT}/preview.html")

Page available at: http://127.0.0.1:8898/preview.html
